# 1. Data Import and Overview
In this notebook, we downloaded the raw data and performed an initial data exploration.

### Import Packages

In [1]:
# Import all necessary packages
import os
import IPython
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import qiime2 as q2
from qiime2 import Visualization
%matplotlib inline

### Set Working Directory
Ensure that the working directory is correctly set to the 'scripts' folder within the main project directory. 
Otherwise, the file paths used in this notebook may not work properly.

In [2]:
# The working directory should normally default to the 'scripts' folder. 
# If it doesn't, set it manually using the command below.
# os.chdir("/home/jovyan/MicrobiomeAnalysis_TummyTribe/scripts")  # Adjust this path to match your folder structure.

# Verify that your working directory is the 'scripts' folder inside the main project directory (.../MicrobiomeAnalysis_TummyTribe/scripts)
cwd = os.getcwd()
if not cwd.endswith("MicrobiomeAnalysis_TummyTribe/scripts"):
    print("WARNING: The working directory is not set to the 'scripts' folder inside 'MicrobiomeAnalysis_TummyTribe'!")
    print("Current working directory:", cwd)
    print("Please set it manually using os.chdir().")
else:
    print(f"Working directory is correctly set to the 'scripts' folder (\"{cwd}\").")

Working directory is correctly set to the 'scripts' folder ("/home/jovyan/MicrobiomeAnalysis_TummyTribe/scripts").


In [3]:
# Data directory for the raw data
data_dir = "../data/raw"
denoising_data_dir = "../data/processed/denoising"

The dataset contains 357 samples from 190 children. Only 39 children have data for all three ages and some children have multiple samples for the same age.
### Duplicates?
Have a look at the ones where there is more than one sample at the same age_months for the same host.

In [4]:
# Import metadata
meta = pd.read_csv(f"{data_dir}/metadata.tsv", sep="\t")
meta.head()

,id,host_id,age_months,geo_location_name,delivery_mode,sex,diet_weaning,diet_milk,treatment_exposure
0,SRR8118533,E000823,4.0,Finland,vaginal,male,no,bd,False
1,SRR8118537,E000823,7.0,Finland,vaginal,male,yes,mixed,False
2,SRR8118564,E001958,4.0,Finland,vaginal,female,yes,bd,False
3,SRR8118650,E001958,7.0,Finland,vaginal,female,yes,mixed,False
4,SRR8118652,E001958,10.0,Finland,vaginal,female,yes,mixed,False


In [5]:
# Group samples per child
metadata = meta.copy() # copy, to not modify the original dataframe
host_ids = meta["host_id"].unique()
samples_per_child = []
for child in host_ids:
    child_data = meta[meta["host_id"] == child]
    # Create a list with all sample ages per child
    sample_ages = child_data["age_months"].to_list() 
    # Count total number of samples per child
    number_samples = len(sample_ages) 
    # Count unique ages per child
    number_unique_ages = len(set(sample_ages))  
    samples_per_child.append({"host_id": child, "sample_ages": sample_ages, "number_samples": number_samples, "number_unique_ages": number_unique_ages})
samples_per_child = pd.DataFrame(samples_per_child)
print("Samples per child: \n", samples_per_child.head())
counts = samples_per_child["number_samples"].value_counts().sort_index()
unique_age_counts = samples_per_child["number_unique_ages"].value_counts().sort_index()
print(f"Number of samples per child: \n{counts}\n")
print(f"Number of samples with unique ages: \n{unique_age_counts}")

Samples per child: 
    host_id       sample_ages  number_samples  number_unique_ages
0  E000823        [4.0, 7.0]               2                   2
1  E001958  [4.0, 7.0, 10.0]               3                   3
2  E002338       [7.0, 10.0]               2                   2
3  E002681             [7.0]               1                   1
4  E003188       [7.0, 10.0]               2                   2
Number of samples per child: 
number_samples
1    72
2    72
3    43
4     3
Name: count, dtype: int64

Number of samples with unique ages: 
number_unique_ages
1    77
2    74
3    39
Name: count, dtype: int64


In [6]:
# Have a look at all hosts with the same age twice
# Count occurrences of each (host_id, age_months) combination
age_counts = meta.groupby(['host_id', 'age_months']).size().reset_index(name='count')

# Find rows where the same age occurs more than once for a child
duplicates = age_counts[age_counts['count'] > 1]
# Merge back to get the actual sample_ids that should be filtered out
samples_to_remove = meta.merge(duplicates[['host_id', 'age_months']], 
                               on=['host_id', 'age_months'], 
                               how='inner')

samples_to_remove

,id,host_id,age_months,geo_location_name,delivery_mode,sex,diet_weaning,diet_milk,treatment_exposure
0,SRR8118930,E004898,4.0,Finland,vaginal,male,yes,bd,True
1,SRR8118932,E004898,4.0,Finland,vaginal,male,yes,mixed,True
2,SRR8118818,E011878,4.0,Finland,vaginal,male,no,bd,True
3,SRR8119238,E011878,4.0,Finland,vaginal,male,no,bd,True
4,SRR8119393,E013094,10.0,Finland,vaginal,male,yes,NaN,False
5,SRR8119392,E013094,10.0,Finland,vaginal,male,yes,NaN,False
6,SRR8119771,E016426,7.0,Finland,vaginal,male,yes,fd,True
7,SRR8119122,E016426,7.0,Finland,vaginal,male,yes,fd,True
8,SRR8119075,E017497,4.0,Finland,vaginal,male,yes,bd,False
9,SRR8119597,E017497,4.0,Finland,vaginal,male,yes,mixed,False


## Feature Table
Get the feature table for these samples as a pandas dataframe and then have a look at the ones with the same age twice for one host.

In [7]:
# Import the feature table as a pandas dataframe
feature_table = q2.Artifact.load(f'{denoising_data_dir}/dada2_table.qza').view(pd.DataFrame)
feature_table.head()

/opt/conda/envs/checkm/lib/python3.10/site-packages/q2_checkm/checkm.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


,35ffcc3b809d667286737d79670b8de5,d46e2205f0c6ecf67b51f83d111c509c,2c982937754e6321f861027032db80f7,99deb3c5ecb022ec05609ebd1112a557,fd44d4cb468fd7dc9b3227867714ed87,2c5e87b291147c04e1b8d1c808b1aee0,c24e0e391aa836b5eae25567c7eb89ee,9908fffab7ed4f3bec44cda2f5084d49,315ca0a729f126b941ba111a16d4d97a,945184b6386c192c0066e0a98a154780,...,2bce8615134b73991ef0e14272646791,3a5317c713023cb528a3d6ea8edaf0a3,0a764c782ff16c18e0e2bde8d46d4608,74cd5ddaf80f3aabb34845c1558d1e54,98d918d67a235038cddd3e27015d9f39,f995b00628c62422bfba901c52048e10,ef500a1b2cafbc7ea19d914cc368cba0,6871fd2ae0f5548e5b16e9cdd0f767ac,919f2dcd2712ba221a695b4a98c5651c,efaa636ca54aed7c824dd9f521028c6b
SRR8115637,12850.0,12487.0,0.0,0.0,0.0,0.0,190.0,1903.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
SRR8115646,2488.0,500.0,72.0,5071.0,0.0,901.0,1071.0,0.0,9365.0,381.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
SRR8115666,387.0,1145.0,0.0,9942.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
SRR8115682,5982.0,333.0,0.0,0.0,0.0,37.0,204.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
SRR8115685,7066.0,0.0,0.0,0.0,0.0,0.0,221.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
data = pd.merge(meta, feature_table, left_on = 'id', right_index = True)
data.head()

,id,host_id,age_months,geo_location_name,delivery_mode,sex,diet_weaning,diet_milk,treatment_exposure,35ffcc3b809d667286737d79670b8de5,...,2bce8615134b73991ef0e14272646791,3a5317c713023cb528a3d6ea8edaf0a3,0a764c782ff16c18e0e2bde8d46d4608,74cd5ddaf80f3aabb34845c1558d1e54,98d918d67a235038cddd3e27015d9f39,f995b00628c62422bfba901c52048e10,ef500a1b2cafbc7ea19d914cc368cba0,6871fd2ae0f5548e5b16e9cdd0f767ac,919f2dcd2712ba221a695b4a98c5651c,efaa636ca54aed7c824dd9f521028c6b
0,SRR8118533,E000823,4.0,Finland,vaginal,male,no,bd,False,1012.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,SRR8118537,E000823,7.0,Finland,vaginal,male,yes,mixed,False,1409.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,SRR8118564,E001958,4.0,Finland,vaginal,female,yes,bd,False,3568.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,SRR8118650,E001958,7.0,Finland,vaginal,female,yes,mixed,False,5386.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,SRR8118652,E001958,10.0,Finland,vaginal,female,yes,mixed,False,6802.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
# Have a look at all hosts with the same age twice
# Count occurrences of each (host_id, age_months) combination
age_counts = data.groupby(['host_id', 'age_months']).size().reset_index(name='count')

# Find rows where the same age occurs more than once for a child
duplicates = age_counts[age_counts['count'] > 1]
# Merge back to get the actual sample_ids that should be filtered out
samples_to_remove = data.merge(duplicates[['host_id', 'age_months']], 
                               on=['host_id', 'age_months'], 
                               how='inner')

samples_to_remove

,id,host_id,age_months,geo_location_name,delivery_mode,sex,diet_weaning,diet_milk,treatment_exposure,35ffcc3b809d667286737d79670b8de5,...,2bce8615134b73991ef0e14272646791,3a5317c713023cb528a3d6ea8edaf0a3,0a764c782ff16c18e0e2bde8d46d4608,74cd5ddaf80f3aabb34845c1558d1e54,98d918d67a235038cddd3e27015d9f39,f995b00628c62422bfba901c52048e10,ef500a1b2cafbc7ea19d914cc368cba0,6871fd2ae0f5548e5b16e9cdd0f767ac,919f2dcd2712ba221a695b4a98c5651c,efaa636ca54aed7c824dd9f521028c6b
0,SRR8118930,E004898,4.0,Finland,vaginal,male,yes,bd,True,4238.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,SRR8118932,E004898,4.0,Finland,vaginal,male,yes,mixed,True,2830.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,SRR8118818,E011878,4.0,Finland,vaginal,male,no,bd,True,11027.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,SRR8119238,E011878,4.0,Finland,vaginal,male,no,bd,True,17791.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,SRR8119393,E013094,10.0,Finland,vaginal,male,yes,NaN,False,2118.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,SRR8119392,E013094,10.0,Finland,vaginal,male,yes,NaN,False,3071.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,SRR8119771,E016426,7.0,Finland,vaginal,male,yes,fd,True,346.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,SRR8119122,E016426,7.0,Finland,vaginal,male,yes,fd,True,738.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,SRR8119075,E017497,4.0,Finland,vaginal,male,yes,bd,False,1695.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,SRR8119597,E017497,4.0,Finland,vaginal,male,yes,mixed,False,444.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# Ignore this code

In [10]:
# Export ('unzip') the Feature Table
! qiime tools export \
  --input-path $denoising_data_dir/dada2_table.qza \
  --output-path $denoising_data_dir/feature-table

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Exported ../data/processed/denoising/dada2_table.qza as BIOMV210DirFmt to directory ../data/processed/denoising/feature-table


In [11]:
# Convert the feature table into a tsv
! biom convert \
  -i $denoising_data_dir/feature-table/feature-table.biom \
  -o $denoising_data_dir/feature-table/feature-table.tsv \
  --to-tsv

In [12]:
# Import the feature table as a pandas dataframe
# Since the converting adds "# Constructed from biom file" as a first line, this line has to be skiped, otherwise the read_csv will fail
feature_table = pd.read_csv(f"{denoising_data_dir}/feature-table/feature-table.tsv", sep="\t", skiprows=1)
feature_table = feature_table.rename(columns={"#OTU ID": "ASV"})
feature_table = feature_table.set_index("ASV")
feature_table = feature_table.T
feature_table.head()

ASV,35ffcc3b809d667286737d79670b8de5,d46e2205f0c6ecf67b51f83d111c509c,2c982937754e6321f861027032db80f7,99deb3c5ecb022ec05609ebd1112a557,fd44d4cb468fd7dc9b3227867714ed87,2c5e87b291147c04e1b8d1c808b1aee0,c24e0e391aa836b5eae25567c7eb89ee,9908fffab7ed4f3bec44cda2f5084d49,315ca0a729f126b941ba111a16d4d97a,945184b6386c192c0066e0a98a154780,...,2bce8615134b73991ef0e14272646791,3a5317c713023cb528a3d6ea8edaf0a3,0a764c782ff16c18e0e2bde8d46d4608,74cd5ddaf80f3aabb34845c1558d1e54,98d918d67a235038cddd3e27015d9f39,f995b00628c62422bfba901c52048e10,ef500a1b2cafbc7ea19d914cc368cba0,6871fd2ae0f5548e5b16e9cdd0f767ac,919f2dcd2712ba221a695b4a98c5651c,efaa636ca54aed7c824dd9f521028c6b
SRR8115637,12850.0,12487.0,0.0,0.0,0.0,0.0,190.0,1903.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
SRR8115646,2488.0,500.0,72.0,5071.0,0.0,901.0,1071.0,0.0,9365.0,381.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
SRR8115666,387.0,1145.0,0.0,9942.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
SRR8115682,5982.0,333.0,0.0,0.0,0.0,37.0,204.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
SRR8115685,7066.0,0.0,0.0,0.0,0.0,0.0,221.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
